In [1]:
from sqlalchemy import text, select
from typing import Optional, Any, Iterable
from sqlalchemy.orm import Session, sessionmaker
from pathlib import Path
import re
import json

from lib.db.database import engine, Base  
from lib.db.models import *

from lib.helpers.normalizeText import normalize_text
from lib.helpers.normalizeDoi import normalize_doi  

In [2]:
with engine.connect() as connection:
    result = connection.execute(text("SELECT 1"))
    print("Conexão OK:", result.scalar())

2026-04-23 15:02:06,958 INFO sqlalchemy.engine.Engine SELECT DATABASE()
2026-04-23 15:02:06,959 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-04-23 15:02:06,965 INFO sqlalchemy.engine.Engine SELECT @@sql_mode
2026-04-23 15:02:06,966 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-04-23 15:02:06,967 INFO sqlalchemy.engine.Engine SELECT @@lower_case_table_names
2026-04-23 15:02:06,968 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-04-23 15:02:06,970 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-23 15:02:06,971 INFO sqlalchemy.engine.Engine SELECT 1
2026-04-23 15:02:06,971 INFO sqlalchemy.engine.Engine [generated in 0.00098s] {}
Conexão OK: 1
2026-04-23 15:02:06,973 INFO sqlalchemy.engine.Engine ROLLBACK


In [ ]:
Base.metadata.create_all(bind=engine)

In [5]:
SessionLocal = sessionmaker(
    bind=engine,
    autoflush=False,
    autocommit=False
)
session = SessionLocal()

In [ ]:
stm = select(Author)
autores = session.execute(stm).scalars().all()
len(autores)

In [ ]:
stm = select(Author)
autores = session.execute(stm).scalars().all()
len(autores)

In [7]:
artigos = []
with open("data/artigos/artigos_normalizados.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        artigo = json.loads(line)
        artigos.append(artigo)
len(artigos)

258

# Helpes

In [32]:

def normalize_issn(value: Optional[str]) -> Optional[str]:
    value = clean_text(value)
    if not value:
        return None

    value = value.upper().replace(" ", "")
    value = value.replace("–", "-").replace("—", "-")

    # mantém só dígitos, X e hífen
    value = re.sub(r"[^0-9X-]", "", value)

    # se vier sem hífen e com 8 caracteres, formata XXXX-XXXX
    raw = value.replace("-", "")
    if len(raw) == 8:
        value = f"{raw[:4]}-{raw[4:]}"
    return value


def normalize_isbn(value: Optional[str]) -> Optional[str]:
    value = clean_text(value)
    if not value:
        return None

    value = value.upper().replace(" ", "").replace("-", "")
    value = re.sub(r"[^0-9X]", "", value)
    return value or None







In [ ]:
def ingest_crossref_message(session, message: dict) -> Publication:
    norm = normalize_crossref_message(message)

    # 1. container
    container = None
    if norm["container"]:
        container = get_or_create_container(session, norm["container"])

    # 2. publication
    publication = upsert_publication(session, norm["publication"], container)

    # 3. contributors
    replace_publication_contributors(session, publication, norm["contributors"])

    # 4. funders
    replace_publication_funders(session, publication, norm["funders"])

    # 5. keywords
    replace_publication_keywords(session, publication, norm["keywords"])

    # 6. métricas
    create_metric_snapshot(session, publication, norm["metric_snapshot"])

    # 7. references
    replace_publication_references(session, publication, norm["references"])

    session.flush()
    return publication

# Lattes

In [ ]:
from lib.db.crud.profile import get_or_create_profile

lattes_id = '2747150211073176'
author_db = get_or_create_profile(session, lattes_id)

In [ ]:
from lib.db.crud.container import get_or_create_container
from lib.db.crud.publication import get_or_create_publication
from lib.db.crud._authors import vinculate_authors_to_publication

In [ ]:
def ingest_article_crossref(session: Session, profile: Author) -> Author:
    
    lattes_id = profile.lattes_id
    path_root = Path('data/curriculos')
    path_cv = Path(path_root / lattes_id)
    path_article = Path(path_cv / "article_crossref.jsonl")
    if not path_article.exists():
        raise FileNotFoundError(f"O arquivo {path_article} não foi encontrado.") 
    artigos = []
    with open(path_article, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            artigo = json.loads(line)
            publication = artigo['publication']
            publication_db = get_or_create_publication(session, publication)
            # container = artigo['container']
            # container_db = get_or_create_container(session, container)
            # publication_db.container = container_db
            created_links = vinculate_authors_to_publication(session, publication_db, artigo['contributors'])
            session.flush()
            artigos.append(artigo)
    
    return artigos
    
artigos = ingest_article_crossref(session, author_db)

In [ ]:
artigo = artigos[0]
publication = artigo['publication']
publication_db = get_or_create_publication(session, publication)
publication_db.id

In [ ]:
container = artigo['container']
container_db = get_or_create_container(session, container)
container_db.id

In [11]:
stm = select(Publication)
publicacoes = session.execute(stm).scalars().all()
len(publicacoes)

2026-04-11 20:10:50,335 INFO sqlalchemy.engine.Engine SELECT publications.id, publications.publication_type, publications.title, publications.subtitle, publications.alternative_title, publications.abstract, publications.date_published, publications.language, publications.subject, publications.doi, publications.isbn, publications.identifier, publications.publisher, publications.url, publications.license, publications.conditions_of_access, publications.is_accessible_for_free, publications.is_part_of_id, publications.page_start, publications.page_end, publications.volume_number, publications.issue_number, publications.edition, publications.number_of_pages, publications.in_support_of, publications.source_organization, publications.source, publications.raw_json, publications.created_at, publications.updated_at 
FROM publications
2026-04-11 20:10:50,340 INFO sqlalchemy.engine.Engine [generated in 0.00434s] {}


258

# Publication container

In [37]:
with open('data/artigos/normalized.json', 'r', encoding='utf-8') as f:
    norm = json.load(f)

In [46]:
with open('data/artigos/normalized.json', 'w', encoding='utf-8') as f:
    json.dump(norm, f, ensure_ascii=False, indent=4)

In [ ]:
from lib.db.crud._authors import vinculate_authors_to_publication
from lib.db.crud.container import get_or_create_container
from lib.db.crud.funders import replace_publication_funders
from lib.db.crud.publication import upsert_publication

def ingest_article(session: Session, artigo: dict):
    container = get_or_create_container(session, artigo['container'])
    publication = upsert_publication(session, artigo['publication'], container)
    created_links = vinculate_authors_to_publication(session, publication, artigo['contributors'])
    # replace_publication_funders(session, publication, artigo['funders'])
    # replace_publication_keywords(session, publication, artigo['keywords'])
    # create_metric_snapshot(session, publication, artigo['metric_snapshot'])
    # replace_publication_references(session, publication, artigo['references'])
    session.flush()
    session.commit()
    return publication

In [9]:
total = session.scalar(select(func.count(Publication.id)))
total

2026-04-09 16:43:09,614 INFO sqlalchemy.engine.Engine SELECT count(publications.id) AS count_1 
FROM publications
2026-04-09 16:43:09,616 INFO sqlalchemy.engine.Engine [cached since 158.5s ago] {}


258

In [12]:
authors = session.scalar(
    select(func.count()).select_from(Author)
)
authors

2026-04-09 16:45:28,262 INFO sqlalchemy.engine.Engine SELECT count(*) AS count_1 
FROM authors
2026-04-09 16:45:28,267 INFO sqlalchemy.engine.Engine [generated in 0.00545s] {}


1236

In [13]:
from sqlalchemy import distinct


stm = (select(
    Author.id,
    Author.full_name,
    func.count(distinct(PublicationContributor.publication_id)).label("publication_count")
)
       .join(PublicationContributor, PublicationContributor.author_id == Author.id)
       .where(PublicationContributor.role == "author")
       .group_by(Author.id, Author.full_name)
       .order_by(func.count(distinct(PublicationContributor.publication_id)).desc())
       )

In [21]:
with open('data/artigo.json', 'w', encoding='utf-8') as f:
    json.dump(artigo, f, ensure_ascii=False, indent=4)

In [ ]:
for artigo in artigos:
    publi = ingest_article(session, artigo)
    print(f"Ingesting article: {publi.title}")

In [34]:
with open('data/artigo.json', 'w', encoding='utf-8') as f:
    json.dump(artigo, f, ensure_ascii=False, indent=4)

In [44]:
lista_de_afiliacoes = artigo['contributors'][0]['contributor']['raw_affiliation']
raw_affiliation = " | ".join(lista_de_afiliacoes)
raw_affiliation

'Department of Zoology University of British Columbia Vancouver British Columbia Canada | Department of Biology McMaster University Hamilton Ontario Canada | Department of Marine Biology and Ecology University of Miami Rosenstiel School of Marine, Atmospheric, and Earth Science Miami Florida USA'

In [13]:
container = get_or_create_container(session, artigo['container'])
container.id

2026-04-09 16:00:28,665 INFO sqlalchemy.engine.Engine SELECT publication_containers.id, publication_containers.name, publication_containers.alternate_name, publication_containers.publisher, publication_containers.issn_print, publication_containers.issn_electronic, publication_containers.isbn, publication_containers.url 
FROM publication_containers 
WHERE publication_containers.issn_print = %(issn_print_1)s
2026-04-09 16:00:28,666 INFO sqlalchemy.engine.Engine [generated in 0.00171s] {'issn_print_1': '1050-4648'}
Encontrado container com o mesmo ISSN print: 1050-4648.


1

In [14]:
def upsert_publication(
    session: Session,
    publication_data: dict,
    container: Optional[PublicationContainer] = None,
) -> Publication:
    """
    Faz upsert de Publication com a seguinte estratégia de matching:

    1. DOI
    2. identifier
    3. fallback por combinação:
       - name
       - publication_type
       - date_published

    Regras:
    - DOI é normalizado antes da busca
    - container é associado se informado
    - campos existentes não são sobrescritos à toa
    - raw_json pode ser atualizado com o payload mais recente
    """

    if not publication_data:
        raise ValueError("publication_data não pode ser vazio")

    title = publication_data.get("title")
    doi = publication_data.get("doi")
    identifier = publication_data.get("identifier")
    publication_type = publication_data.get("publication_type")
    date_published = publication_data.get("date_published")
    

    if not title and not doi and not identifier:
        raise ValueError(
            "publication_data precisa ter ao menos 'title', 'doi' ou 'identifier'"
        )

    # # 1. match por DOI
    if doi:
        existing = session.scalar(
            select(Publication).where(Publication.doi == doi)
        )
        if existing:
            publication_data = dict(publication_data)
            publication_data["doi"] = doi
            # _update_publication_if_needed(
            #     existing,
            #     publication_data=publication_data,
            #     container=container,
            # )
            session.flush()
            return existing

    # # 2. match por identifier
    if identifier:
        existing = session.scalar(
            select(Publication).where(Publication.identifier == identifier)
        )
        if existing:
            publication_data = dict(publication_data)
            publication_data["doi"] = doi
            # _update_publication_if_needed(
            #     existing,
            #     publication_data=publication_data,
            #     container=container,
            # )
            session.flush()
            return existing

    # 3. fallback por name + publication_type + date_published
    if title:
        normalized_title = normalize_text(title)

        candidates = session.scalars(
            select(Publication).where(Publication.title.is_not(None))
        ).all()

        for candidate in candidates:
            if normalize_text(candidate.title) != normalized_title:
                continue

            if publication_type and candidate.publication_type:
                if candidate.publication_type != publication_type:
                    continue

            if date_published and candidate.date_published:
                if not (candidate.date_published == date_published):
                    continue

            publication_data = dict(publication_data)
            publication_data["doi"] = doi
            # _update_publication_if_needed(
            #     candidate,
            #     publication_data=publication_data,
            #     container=container,
            # )
            session.flush()
            return candidate

    # se não encontrou, cria
    print("Criando publicação:", title)
    publication = Publication(**publication_data)

    if container is not None:
        publication.container = container

    session.add(publication)
    session.flush()
    session.commit()
    
    return publication

In [15]:
publication = upsert_publication(session, artigo['publication'], container)
publication.id

2026-04-09 16:01:49,159 INFO sqlalchemy.engine.Engine SELECT publications.id, publications.publication_type, publications.title, publications.subtitle, publications.alternative_title, publications.abstract, publications.date_published, publications.language, publications.subject, publications.doi, publications.isbn, publications.identifier, publications.publisher, publications.url, publications.license, publications.conditions_of_access, publications.is_accessible_for_free, publications.is_part_of_id, publications.page_start, publications.page_end, publications.volume_number, publications.issue_number, publications.edition, publications.number_of_pages, publications.in_support_of, publications.source_organization, publications.source, publications.raw_json, publications.created_at, publications.updated_at 
FROM publications 
WHERE publications.doi = %(doi_1)s
2026-04-09 16:01:49,161 INFO sqlalchemy.engine.Engine [generated in 0.00215s] {'doi_1': '10.1016/j.fsi.2025.110959'}


1

# Publication

In [43]:
publication_data = norm["publication"]
publication = upsert_publication(session, publication_data, container)

2026-04-07 16:20:18,849 INFO sqlalchemy.engine.Engine SELECT publications.id, publications.publication_type, publications.title, publications.subtitle, publications.alternative_title, publications.abstract, publications.date_published, publications.language, publications.subject, publications.doi, publications.isbn, publications.identifier, publications.publisher, publications.url, publications.license, publications.conditions_of_access, publications.is_accessible_for_free, publications.is_part_of_id, publications.page_start, publications.page_end, publications.volume_number, publications.issue_number, publications.edition, publications.number_of_pages, publications.in_support_of, publications.source_organization, publications.source, publications.raw_json, publications.created_at, publications.updated_at 
FROM publications 
WHERE publications.doi = %(doi_1)s
2026-04-07 16:20:18,853 INFO sqlalchemy.engine.Engine [generated in 0.00376s] {'doi_1': '10.1016/j.fsi.2025.110959'}


# Contributors

In [17]:
contributors = norm["contributors"]
contributor = contributors[0]
author_data = contributor.get("author")
contributor = contributor.get("contributor")
contributor

NameError: name 'norm' is not defined

In [ ]:
import re
from typing import Optional

from sqlalchemy import select
from sqlalchemy.orm import Session

from models import Publication, PublicationContainer


def clean_text(value: Optional[str]) -> Optional[str]:
    if value is None:
        return None
    value = str(value).strip()
    return value or None


def normalize_text(value: Optional[str]) -> Optional[str]:
    value = clean_text(value)
    if not value:
        return None
    value = re.sub(r"\s+", " ", value)
    return value.strip().lower()








def _should_replace_text(old_value: Optional[str], new_value: Optional[str]) -> bool:
    """
    Regras simples:
    - se o antigo está vazio e o novo não, substitui
    - se ambos existem, pode substituir apenas se o novo parece mais completo
    """
    if not new_value:
        return False
    if not old_value:
        return True

    old_value = old_value.strip()
    new_value = new_value.strip()

    if old_value == new_value:
        return False

    # prefere o texto maior em campos descritivos
    return len(new_value) > len(old_value)


def _update_publication_if_needed(
    publication: Publication,
    *,
    publication_data: dict,
    container: Optional[PublicationContainer],
) -> None:
    """
    Atualiza somente campos vazios ou, em alguns casos, quando o novo valor
    parece mais completo.
    """

    # Identidade e tipagem
    if not publication.publication_type and publication_data.get("publication_type"):
        publication.publication_type = publication_data.get("publication_type")

    if not publication.status and publication_data.get("status"):
        publication.status = publication_data.get("status")

    # Títulos e descrição
    if _should_replace_text(publication.name, publication_data.get("name")):
        publication.name = publication_data.get("name")

    if _should_replace_text(publication.headline, publication_data.get("headline")):
        publication.headline = publication_data.get("headline")

    if _should_replace_text(
        publication.alternative_headline,
        publication_data.get("alternative_headline"),
    ):
        publication.alternative_headline = publication_data.get("alternative_headline")

    if _should_replace_text(publication.abstract, publication_data.get("abstract")):
        publication.abstract = publication_data.get("abstract")

    # Datas
    if not publication.date_published and publication_data.get("date_published"):
        publication.date_published = publication_data.get("date_published")

    # Idioma e classificação
    if not publication.in_language and publication_data.get("in_language"):
        publication.in_language = publication_data.get("in_language")

    if not publication.genre and publication_data.get("genre"):
        publication.genre = publication_data.get("genre")

    if _should_replace_text(
        publication.keywords_text, publication_data.get("keywords_text")
    ):
        publication.keywords_text = publication_data.get("keywords_text")

    # Identificadores
    normalized_new_doi = normalize_doi(publication_data.get("doi"))
    if not publication.doi and normalized_new_doi:
        publication.doi = normalized_new_doi

    if not publication.isbn and publication_data.get("isbn"):
        publication.isbn = publication_data.get("isbn")

    if not publication.identifier and publication_data.get("identifier"):
        publication.identifier = publication_data.get("identifier")

    # Publicação / acesso
    if not publication.publisher and publication_data.get("publisher"):
        publication.publisher = publication_data.get("publisher")

    if not publication.url and publication_data.get("url"):
        publication.url = publication_data.get("url")

    if not publication.license and publication_data.get("license"):
        publication.license = publication_data.get("license")

    if _should_replace_text(
        publication.conditions_of_access,
        publication_data.get("conditions_of_access"),
    ):
        publication.conditions_of_access = publication_data.get("conditions_of_access")

    if publication.is_accessible_for_free is None and (
        publication_data.get("is_accessible_for_free") is not None
    ):
        publication.is_accessible_for_free = publication_data.get(
            "is_accessible_for_free"
        )

    # Container
    if container is not None and publication.is_part_of_id != container.id:
        publication.container = container

    # Paginação / localização
    if not publication.page_start and publication_data.get("page_start"):
        publication.page_start = publication_data.get("page_start")

    if not publication.page_end and publication_data.get("page_end"):
        publication.page_end = publication_data.get("page_end")

    if not publication.volume_number and publication_data.get("volume_number"):
        publication.volume_number = publication_data.get("volume_number")

    if not publication.issue_number and publication_data.get("issue_number"):
        publication.issue_number = publication_data.get("issue_number")

    if not publication.edition and publication_data.get("edition"):
        publication.edition = publication_data.get("edition")

    if not publication.number_of_pages and publication_data.get("number_of_pages"):
        publication.number_of_pages = publication_data.get("number_of_pages")

    # Teses / dissertações
    if not publication.in_support_of and publication_data.get("in_support_of"):
        publication.in_support_of = publication_data.get("in_support_of")

    if not publication.source_organization and publication_data.get("source_organization"):
        publication.source_organization = publication_data.get("source_organization")

    # Proveniência
    if not publication.source and publication_data.get("source"):
        publication.source = publication_data.get("source")

    # raw_json:
    # aqui faz sentido substituir sempre que houver um payload mais novo/canônico
    if publication_data.get("raw_json") is not None:
        publication.raw_json = publication_data.get("raw_json")


def upsert_publication(
    session: Session,
    publication_data: dict,
    container: Optional[PublicationContainer] = None,
) -> Publication:
    """
    Faz upsert de Publication com a seguinte estratégia de matching:

    1. DOI
    2. identifier
    3. fallback por combinação:
       - name
       - publication_type
       - date_published

    Regras:
    - DOI é normalizado antes da busca
    - container é associado se informado
    - campos existentes não são sobrescritos à toa
    - raw_json pode ser atualizado com o payload mais recente
    """

    if not publication_data:
        raise ValueError("publication_data não pode ser vazio")

    name = clean_text(publication_data.get("name"))
    normalized_doi = normalize_doi(publication_data.get("doi"))
    identifier = clean_text(publication_data.get("identifier"))
    publication_type = clean_text(publication_data.get("publication_type"))
    date_published = publication_data.get("date_published")

    if not name and not normalized_doi and not identifier:
        raise ValueError(
            "publication_data precisa ter ao menos 'name', 'doi' ou 'identifier'"
        )

    # 1. match por DOI
    if normalized_doi:
        existing = session.scalar(
            select(Publication).where(Publication.doi == normalized_doi)
        )
        if existing:
            publication_data = dict(publication_data)
            publication_data["doi"] = normalized_doi
            _update_publication_if_needed(
                existing,
                publication_data=publication_data,
                container=container,
            )
            session.flush()
            return existing

    # 2. match por identifier
    if identifier:
        existing = session.scalar(
            select(Publication).where(Publication.identifier == identifier)
        )
        if existing:
            publication_data = dict(publication_data)
            publication_data["doi"] = normalized_doi
            _update_publication_if_needed(
                existing,
                publication_data=publication_data,
                container=container,
            )
            session.flush()
            return existing

    # 3. fallback por name + publication_type + date_published
    if name:
        normalized_name = normalize_text(name)

        candidates = session.scalars(
            select(Publication).where(Publication.name.is_not(None))
        ).all()

        for candidate in candidates:
            if normalize_text(candidate.name) != normalized_name:
                continue

            if publication_type and candidate.publication_type:
                if candidate.publication_type != publication_type:
                    continue

            if date_published and candidate.date_published:
                if not _same_date(candidate.date_published, date_published):
                    continue

            publication_data = dict(publication_data)
            publication_data["doi"] = normalized_doi
            _update_publication_if_needed(
                candidate,
                publication_data=publication_data,
                container=container,
            )
            session.flush()
            return candidate

    # se não encontrou, cria
    publication = Publication(
        publication_type=publication_type,
        status=clean_text(publication_data.get("status")),
        name=name or "Sem título",
        headline=clean_text(publication_data.get("headline")),
        alternative_headline=clean_text(publication_data.get("alternative_headline")),
        abstract=clean_text(publication_data.get("abstract")),
        date_published=date_published,
        in_language=clean_text(publication_data.get("in_language")),
        genre=clean_text(publication_data.get("genre")),
        keywords_text=clean_text(publication_data.get("keywords_text")),
        doi=normalized_doi,
        isbn=clean_text(publication_data.get("isbn")),
        identifier=identifier,
        publisher=clean_text(publication_data.get("publisher")),
        url=clean_text(publication_data.get("url")),
        license=clean_text(publication_data.get("license")),
        conditions_of_access=clean_text(publication_data.get("conditions_of_access")),
        is_accessible_for_free=publication_data.get("is_accessible_for_free"),
        page_start=clean_text(publication_data.get("page_start")),
        page_end=clean_text(publication_data.get("page_end")),
        volume_number=clean_text(publication_data.get("volume_number")),
        issue_number=clean_text(publication_data.get("issue_number")),
        edition=clean_text(publication_data.get("edition")),
        number_of_pages=publication_data.get("number_of_pages"),
        in_support_of=clean_text(publication_data.get("in_support_of")),
        source_organization=clean_text(publication_data.get("source_organization")),
        source=clean_text(publication_data.get("source")),
        raw_json=publication_data.get("raw_json"),
    )

    if container is not None:
        publication.container = container

    session.add(publication)
    session.flush()
    return publication

# Keywords

In [ ]:
# 5. keywords
replace_publication_keywords(session, publication, norm["keywords"])

# References

In [50]:
def resolve_cited_publication_id(
    session: Session,
    ref_data: dict[str, Any],
) -> tuple[Optional[int], Optional[str]]:
    """
    Tenta resolver a referência para uma Publication já existente no banco.
    Estratégia atual:
    - match por DOI
    Se encontrar, retorna (publication_id, 'doi')
    Se não encontrar, retorna (None, valor_original_match_source)
    """
    doi = normalize_doi(ref_data.get("doi"))
    original_match_source = ref_data.get("match_source")

    if doi:
        cited = session.scalar(
            select(Publication.id).where(Publication.doi == doi)
        )
        if cited is not None:
            return cited, "doi"

    return None, original_match_source

In [46]:
def replace_publication_references(
    session: Session,
    publication: Publication,
    references_data: list[dict[str, Any]],
) -> None:
    """
    Remove as referências atuais da publicação e recria a lista com base
    no JSON normalizado.

    Isso combina com:
    - UniqueConstraint(citing_publication_id, position)
    - relationship(... cascade='all, delete-orphan')
    """
    publication.outgoing_references.clear()
    session.flush()

    for index, ref_data in enumerate(references_data, start=1):
        cited_publication_id, match_source = resolve_cited_publication_id(
            session, ref_data
        )

        ref = PublicationReference(
            citing_publication=publication,
            cited_publication_id=cited_publication_id,
            position=index,
            doi=normalize_doi(ref_data.get("doi")),
            title=ref_data.get("title"),
            author=ref_data.get("author"),
            journal_title=ref_data.get("journal_title"),
            year=ref_data.get("year"),
            volume=ref_data.get("volume"),
            issue=ref_data.get("issue"),
            match_source=match_source,
        )

        publication.outgoing_references.append(ref)
    session.flush()
    session.commit()
    return publication

In [48]:
references = norm["references"]
reference = references[0]
reference

{'doi': '10.1016/j.ijpara.2020.04.007',
 'title': 'Why ignoring parasites in fish ecology is a mistake',
 'author': 'Timi',
 'journal_title': 'Int. J. Parasitol.',
 'year': '2020',
 'volume': '50',
 'issue': None,
 'match_source': None}

In [51]:
x, y = resolve_cited_publication_id(session, reference)

2026-04-07 16:27:41,705 INFO sqlalchemy.engine.Engine SELECT publications.id 
FROM publications 
WHERE publications.doi = %(doi_1)s
2026-04-07 16:27:41,716 INFO sqlalchemy.engine.Engine [generated in 0.01134s] {'doi_1': '10.1016/j.ijpara.2020.04.007'}


In [54]:
publication = replace_publication_references(session, publication, references)

2026-04-07 16:28:06,215 INFO sqlalchemy.engine.Engine SELECT publications.id 
FROM publications 
WHERE publications.doi = %(doi_1)s
2026-04-07 16:28:06,217 INFO sqlalchemy.engine.Engine [cached since 24.51s ago] {'doi_1': '10.1016/j.ijpara.2020.04.007'}
2026-04-07 16:28:06,226 INFO sqlalchemy.engine.Engine SELECT publications.id 
FROM publications 
WHERE publications.doi = %(doi_1)s
2026-04-07 16:28:06,227 INFO sqlalchemy.engine.Engine [cached since 24.52s ago] {'doi_1': '10.1007/s00436-017-5384-3'}
2026-04-07 16:28:06,230 INFO sqlalchemy.engine.Engine SELECT publications.id 
FROM publications 
WHERE publications.doi = %(doi_1)s
2026-04-07 16:28:06,232 INFO sqlalchemy.engine.Engine [cached since 24.53s ago] {'doi_1': '10.20506/rst.27.2.1820'}
2026-04-07 16:28:06,236 INFO sqlalchemy.engine.Engine SELECT publications.id 
FROM publications 
WHERE publications.doi = %(doi_1)s
2026-04-07 16:28:06,236 INFO sqlalchemy.engine.Engine [cached since 24.53s ago] {'doi_1': '10.1016/bs.apar.2021.03.

In [56]:
publication.outgoing_references

In [35]:
def ingest_publication_with_references(
    session: Session,
    normalized_data: dict[str, Any],
) -> Publication:
    """
    Ingestão da publicação e de suas referências a partir do JSON normalizado.
    """
    pub_data = normalized_data["publication"]
    references_data = normalized_data.get("references", [])

    publication = upsert_publication(session, pub_data)
    session.flush()  # garante publication.id para vincular referências

    replace_publication_references(session, publication, references_data)

    session.flush()
    return publication

In [38]:
pub_data = norm["publication"]
pub_data

{'publication_type': 'journal-article',
 'title': 'Immunometabolic costs of parasitism under warming: Impaired mitochondrial function and thermal tolerance in an Amazonian fish',
 'subtitle': None,
 'alternative_title': None,
 'abstract': None,
 'date_published': '2026-01-01',
 'language': 'en',
 'subject': None,
 'doi': '10.1016/j.fsi.2025.110959',
 'isbn': None,
 'identifier': 'S1050464825008484',
 'publisher': 'Elsevier BV',
 'url': 'https://linkinghub.elsevier.com/retrieve/pii/S1050464825008484',
 'license': 'https://www.elsevier.com/tdm/userlicense/1.0/',
 'conditions_of_access': 'Acesso ao conteúdo sujeito à licença de mineração/texto e dados do editor | Licença: https://www.elsevier.com/tdm/userlicense/1.0/ | Vigência da licença a partir de 2026-01-01 | Links de conteúdo identificados: text/plain, text/xml | URL principal no domínio linkinghub.elsevier.com | Informações adicionais: copyright © 2025 Elsevier Ltd. All rights are reserved, including those for text and data mining, 

In [ ]:
from __future__ import annotations

import json
from datetime import date
from pathlib import Path
from typing import Any, Optional

from sqlalchemy import select
from sqlalchemy.orm import Session

from lib.db.models import Publication, PublicationReference


def parse_date(value: Optional[str]) -> Optional[date]:
    """
    Converte 'YYYY-MM-DD' para date.
    Retorna None para valores vazios ou inválidos.
    """
    if not value:
        return None

    try:
        return date.fromisoformat(value)
    except (TypeError, ValueError):
        return None


def normalize_doi(doi: Optional[str]) -> Optional[str]:
    """
    Normaliza DOI para facilitar matching.
    """
    if not doi:
        return None
    return doi.strip().lower()


def upsert_publication(session: Session, pub_data: dict[str, Any]) -> Publication:
    """
    Cria ou atualiza uma Publication com base principalmente no DOI.
    Se não houver DOI, cria um novo registro.
    """
    doi = normalize_doi(pub_data.get("doi"))
    publication: Optional[Publication] = None

    if doi:
        publication = session.scalar(
            select(Publication).where(Publication.doi == doi)
        )

    if publication is None:
        publication = Publication()
        session.add(publication)

    publication.publication_type = pub_data.get("publication_type")
    publication.title = pub_data.get("title") or ""
    publication.subtitle = pub_data.get("subtitle")
    publication.alternative_title = pub_data.get("alternative_title")
    publication.abstract = pub_data.get("abstract")
    publication.date_published = parse_date(pub_data.get("date_published"))
    publication.language = pub_data.get("language")
    publication.subject = pub_data.get("subject")
    publication.doi = doi
    publication.isbn = pub_data.get("isbn")
    publication.identifier = pub_data.get("identifier")
    publication.publisher = pub_data.get("publisher")
    publication.url = pub_data.get("url")
    publication.license = pub_data.get("license")
    publication.conditions_of_access = pub_data.get("conditions_of_access")
    publication.is_accessible_for_free = pub_data.get("is_accessible_for_free")
    publication.page_start = pub_data.get("page_start")
    publication.page_end = pub_data.get("page_end")
    publication.volume_number = pub_data.get("volume_number")
    publication.issue_number = pub_data.get("issue_number")
    publication.edition = pub_data.get("edition")
    publication.source = pub_data.get("source")
    publication.raw_json = pub_data.get("raw_json")

    return publication











def ingest_publication_with_references_from_file(
    session: Session,
    file_path: str | Path,
) -> Publication:
    """
    Lê um arquivo JSON normalizado do disco e executa a ingestão.
    """
    path = Path(file_path)

    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    return ingest_publication_with_references(session, data)

In [ ]:
from sqlalchemy.orm import Session
from lib.db.database import SessionLocal

with SessionLocal() as session:
    publication = ingest_publication_with_references_from_file(
        session,
        "normalized.json",
    )
    session.commit()
    print(publication.id, publication.doi, publication.title)